In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_test_data
from shufflenet_v2 import ShuffleNetV2
from helper_utils import show_sample_images, training_loop, plot_training_history, visualize_predictions, plot_confusion_matrix

import kd_utils

from fit_net_wrapper import FitNetWrapper

In [2]:
student_train_loader, student_val_loader, student_test_loader = load_train_val_test_data(batch_size=32)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
densenet_201_teacher = timm.create_model("densenet201", pretrained=False, num_classes=10)
densenet_201_teacher.load_state_dict(torch.load("densenet_201_teacher.pth"))

<All keys matched successfully>

# Train with FitNet

In [5]:
hint_epochs = 15
train_epochs = 15

In [6]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.50,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2306
Epoch 2/15 - Hint Loss: 1.7758
Epoch 3/15 - Hint Loss: 1.6196
Epoch 4/15 - Hint Loss: 1.5383
Epoch 5/15 - Hint Loss: 1.4887
Epoch 6/15 - Hint Loss: 1.4550
Epoch 7/15 - Hint Loss: 1.4309
Epoch 8/15 - Hint Loss: 1.4113
Epoch 9/15 - Hint Loss: 1.3968
Epoch 10/15 - Hint Loss: 1.3842
Epoch 11/15 - Hint Loss: 1.3748
Epoch 12/15 - Hint Loss: 1.3673
Epoch 13/15 - Hint Loss: 1.3716
Epoch 14/15 - Hint Loss: 1.3611
Epoch 15/15 - Hint Loss: 1.3554


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.9947%, Train Hard Loss: 1.5445, Train Soft Loss: 0.6135, Train Distill Loss: 5.6806 | Val Hard Loss: 1.3277, Val Soft Loss: 0.3512, Val Distill Loss: 3.4736, Val Acc: 70.2638%
Epoch 2/15 - Train Acc: 76.5289%, Train Hard Loss: 0.8830, Train Soft Loss: 0.3231, Train Distill Loss: 3.0264 | Val Hard Loss: 0.9448, Val Soft Loss: 0.2562, Val Distill Loss: 2.5218, Val Acc: 77.5180%
Epoch 3/15 - Train Acc: 84.2074%, Train Hard Loss: 0.5620, Train Soft Loss: 0.2202, Train Distill Loss: 2.0429 | Val Hard Loss: 1.2307, Val Soft Loss: 0.3135, Val Distill Loss: 3.1234, Val Acc: 72.4820%
Epoch 4/15 - Train Acc: 89.0458%, Train Hard Loss: 0.3808, Train Soft Loss: 0.1689, Train Distill Loss: 1.5417 | Val Hard Loss: 0.7574, Val Soft Loss: 0.2303, Val Distill Loss: 2.2208, Val Acc: 81.1151%
Epoch 5/15 - Train Acc: 92.8775%, Train Hard Loss: 0.2352, Train Soft Loss: 0.1305, Train Distill Loss: 1.1619 | Val Hard Loss: 0.3804, Val Soft Loss: 0.1220, Val Distill Loss: 1.1662, Val

0.935700575815739

In [7]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.55,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2515
Epoch 2/15 - Hint Loss: 1.7862
Epoch 3/15 - Hint Loss: 1.6249
Epoch 4/15 - Hint Loss: 1.5401
Epoch 5/15 - Hint Loss: 1.4866
Epoch 6/15 - Hint Loss: 1.4509
Epoch 7/15 - Hint Loss: 1.4260
Epoch 8/15 - Hint Loss: 1.4067
Epoch 9/15 - Hint Loss: 1.3925
Epoch 10/15 - Hint Loss: 1.3805
Epoch 11/15 - Hint Loss: 1.3715
Epoch 12/15 - Hint Loss: 1.3661
Epoch 13/15 - Hint Loss: 1.3592
Epoch 14/15 - Hint Loss: 1.3548
Epoch 15/15 - Hint Loss: 1.3542


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.4455%, Train Hard Loss: 1.4819, Train Soft Loss: 0.6087, Train Distill Loss: 5.1978 | Val Hard Loss: 2.0402, Val Soft Loss: 0.4999, Val Distill Loss: 5.0195, Val Acc: 60.9712%
Epoch 2/15 - Train Acc: 76.7844%, Train Hard Loss: 0.8560, Train Soft Loss: 0.3211, Train Distill Loss: 2.7827 | Val Hard Loss: 1.2396, Val Soft Loss: 0.3154, Val Distill Loss: 3.1432, Val Acc: 72.1223%
Epoch 3/15 - Train Acc: 85.1390%, Train Hard Loss: 0.5318, Train Soft Loss: 0.2193, Train Distill Loss: 1.8716 | Val Hard Loss: 0.9778, Val Soft Loss: 0.2762, Val Distill Loss: 2.6985, Val Acc: 76.5588%
Epoch 4/15 - Train Acc: 89.9624%, Train Hard Loss: 0.3341, Train Soft Loss: 0.1627, Train Distill Loss: 1.3555 | Val Hard Loss: 0.7312, Val Soft Loss: 0.1924, Val Distill Loss: 1.9045, Val Acc: 80.9353%
Epoch 5/15 - Train Acc: 93.4636%, Train Hard Loss: 0.2042, Train Soft Loss: 0.1249, Train Distill Loss: 1.0113 | Val Hard Loss: 0.3518, Val Soft Loss: 0.1145, Val Distill Loss: 1.0922, Val

0.9371401151631478

In [8]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.60,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2464
Epoch 2/15 - Hint Loss: 1.7989
Epoch 3/15 - Hint Loss: 1.6352
Epoch 4/15 - Hint Loss: 1.5473
Epoch 5/15 - Hint Loss: 1.4928
Epoch 6/15 - Hint Loss: 1.4559
Epoch 7/15 - Hint Loss: 1.4290
Epoch 8/15 - Hint Loss: 1.4087
Epoch 9/15 - Hint Loss: 1.3938
Epoch 10/15 - Hint Loss: 1.3836
Epoch 11/15 - Hint Loss: 1.3737
Epoch 12/15 - Hint Loss: 1.3663
Epoch 13/15 - Hint Loss: 1.3613
Epoch 14/15 - Hint Loss: 1.3567
Epoch 15/15 - Hint Loss: 1.3526


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 58.0766%, Train Hard Loss: 1.4305, Train Soft Loss: 0.6022, Train Distill Loss: 4.7122 | Val Hard Loss: 1.2661, Val Soft Loss: 0.3354, Val Distill Loss: 3.3164, Val Acc: 69.8441%
Epoch 2/15 - Train Acc: 78.4823%, Train Hard Loss: 0.8188, Train Soft Loss: 0.3109, Train Distill Loss: 2.4808 | Val Hard Loss: 0.9649, Val Soft Loss: 0.2721, Val Distill Loss: 2.6590, Val Acc: 74.7602%
Epoch 3/15 - Train Acc: 85.6799%, Train Hard Loss: 0.5323, Train Soft Loss: 0.2181, Train Distill Loss: 1.7155 | Val Hard Loss: 0.7378, Val Soft Loss: 0.2244, Val Distill Loss: 2.1645, Val Acc: 81.4748%
Epoch 4/15 - Train Acc: 89.7671%, Train Hard Loss: 0.3508, Train Soft Loss: 0.1648, Train Distill Loss: 1.2651 | Val Hard Loss: 0.4248, Val Soft Loss: 0.1386, Val Distill Loss: 1.3209, Val Acc: 88.3094%
Epoch 5/15 - Train Acc: 92.8625%, Train Hard Loss: 0.2204, Train Soft Loss: 0.1306, Train Distill Loss: 0.9681 | Val Hard Loss: 0.3964, Val Soft Loss: 0.1229, Val Distill Loss: 1.1814, Val

0.9414587332053743

In [15]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.65,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2556
Epoch 2/15 - Hint Loss: 1.7851
Epoch 3/15 - Hint Loss: 1.6165
Epoch 4/15 - Hint Loss: 1.5319
Epoch 5/15 - Hint Loss: 1.4812
Epoch 6/15 - Hint Loss: 1.4481
Epoch 7/15 - Hint Loss: 1.4226
Epoch 8/15 - Hint Loss: 1.4038
Epoch 9/15 - Hint Loss: 1.3890
Epoch 10/15 - Hint Loss: 1.3778
Epoch 11/15 - Hint Loss: 1.3695
Epoch 12/15 - Hint Loss: 1.3627
Epoch 13/15 - Hint Loss: 1.3569
Epoch 14/15 - Hint Loss: 1.3526
Epoch 15/15 - Hint Loss: 1.3493


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.7092%, Train Hard Loss: 1.4312, Train Soft Loss: 0.6281, Train Distill Loss: 4.4475 | Val Hard Loss: 1.2907, Val Soft Loss: 0.4166, Val Distill Loss: 3.9782, Val Acc: 66.3669%
Epoch 2/15 - Train Acc: 76.6792%, Train Hard Loss: 0.8648, Train Soft Loss: 0.3341, Train Distill Loss: 2.4333 | Val Hard Loss: 0.7920, Val Soft Loss: 0.2631, Val Distill Loss: 2.5005, Val Acc: 77.6978%
Epoch 3/15 - Train Acc: 85.0188%, Train Hard Loss: 0.5439, Train Soft Loss: 0.2260, Train Distill Loss: 1.6190 | Val Hard Loss: 0.5842, Val Soft Loss: 0.1784, Val Distill Loss: 1.7193, Val Acc: 85.3118%
Epoch 4/15 - Train Acc: 90.1728%, Train Hard Loss: 0.3244, Train Soft Loss: 0.1672, Train Distill Loss: 1.1474 | Val Hard Loss: 0.5313, Val Soft Loss: 0.1489, Val Distill Loss: 1.4570, Val Acc: 87.5300%
Epoch 5/15 - Train Acc: 92.6371%, Train Hard Loss: 0.2209, Train Soft Loss: 0.1337, Train Distill Loss: 0.8922 | Val Hard Loss: 0.4671, Val Soft Loss: 0.1327, Val Distill Loss: 1.2955, Val

0.9380998080614203

In [10]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.70,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2337
Epoch 2/15 - Hint Loss: 1.7831
Epoch 3/15 - Hint Loss: 1.6215
Epoch 4/15 - Hint Loss: 1.5373
Epoch 5/15 - Hint Loss: 1.4868
Epoch 6/15 - Hint Loss: 1.4502
Epoch 7/15 - Hint Loss: 1.4240
Epoch 8/15 - Hint Loss: 1.4045
Epoch 9/15 - Hint Loss: 1.3897
Epoch 10/15 - Hint Loss: 1.3783
Epoch 11/15 - Hint Loss: 1.3698
Epoch 12/15 - Hint Loss: 1.3632
Epoch 13/15 - Hint Loss: 1.3614
Epoch 14/15 - Hint Loss: 1.3553
Epoch 15/15 - Hint Loss: 1.3517


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.7844%, Train Hard Loss: 1.4215, Train Soft Loss: 0.6428, Train Distill Loss: 4.0804 | Val Hard Loss: 1.6839, Val Soft Loss: 0.4376, Val Distill Loss: 4.3424, Val Acc: 60.3118%
Epoch 2/15 - Train Acc: 76.1232%, Train Hard Loss: 0.8609, Train Soft Loss: 0.3435, Train Distill Loss: 2.2516 | Val Hard Loss: 0.8817, Val Soft Loss: 0.2870, Val Distill Loss: 2.7370, Val Acc: 75.7194%
Epoch 3/15 - Train Acc: 84.2975%, Train Hard Loss: 0.5564, Train Soft Loss: 0.2340, Train Distill Loss: 1.5125 | Val Hard Loss: 0.6830, Val Soft Loss: 0.2049, Val Distill Loss: 1.9806, Val Acc: 81.0552%
Epoch 4/15 - Train Acc: 89.2562%, Train Hard Loss: 0.3677, Train Soft Loss: 0.1790, Train Distill Loss: 1.1167 | Val Hard Loss: 0.7271, Val Soft Loss: 0.1949, Val Distill Loss: 1.9231, Val Acc: 82.8537%
Epoch 5/15 - Train Acc: 93.2231%, Train Hard Loss: 0.2102, Train Soft Loss: 0.1335, Train Distill Loss: 0.7877 | Val Hard Loss: 0.5711, Val Soft Loss: 0.1837, Val Distill Loss: 1.7553, Val

0.940978886756238

In [11]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.75,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2221
Epoch 2/15 - Hint Loss: 1.7775
Epoch 3/15 - Hint Loss: 1.6156
Epoch 4/15 - Hint Loss: 1.5321
Epoch 5/15 - Hint Loss: 1.4820
Epoch 6/15 - Hint Loss: 1.4480
Epoch 7/15 - Hint Loss: 1.4239
Epoch 8/15 - Hint Loss: 1.4051
Epoch 9/15 - Hint Loss: 1.3912
Epoch 10/15 - Hint Loss: 1.3801
Epoch 11/15 - Hint Loss: 1.3731
Epoch 12/15 - Hint Loss: 1.3645
Epoch 13/15 - Hint Loss: 1.3600
Epoch 14/15 - Hint Loss: 1.3563
Epoch 15/15 - Hint Loss: 1.3531


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 55.7476%, Train Hard Loss: 1.4050, Train Soft Loss: 0.6496, Train Distill Loss: 3.6520 | Val Hard Loss: 1.0808, Val Soft Loss: 0.3818, Val Distill Loss: 3.5944, Val Acc: 68.1055%
Epoch 2/15 - Train Acc: 75.6123%, Train Hard Loss: 0.8811, Train Soft Loss: 0.3650, Train Distill Loss: 2.1208 | Val Hard Loss: 0.8574, Val Soft Loss: 0.2591, Val Distill Loss: 2.5012, Val Acc: 76.9185%
Epoch 3/15 - Train Acc: 82.6296%, Train Hard Loss: 0.6023, Train Soft Loss: 0.2593, Train Distill Loss: 1.4888 | Val Hard Loss: 1.4924, Val Soft Loss: 0.3954, Val Distill Loss: 3.9095, Val Acc: 67.9856%
Epoch 4/15 - Train Acc: 88.5650%, Train Hard Loss: 0.3744, Train Soft Loss: 0.1884, Train Distill Loss: 1.0344 | Val Hard Loss: 0.5555, Val Soft Loss: 0.1711, Val Distill Loss: 1.6467, Val Acc: 85.6715%
Epoch 5/15 - Train Acc: 92.5019%, Train Hard Loss: 0.2383, Train Soft Loss: 0.1504, Train Distill Loss: 0.7804 | Val Hard Loss: 0.4714, Val Soft Loss: 0.1371, Val Distill Loss: 1.3323, Val

0.9366602687140115

In [12]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2110
Epoch 2/15 - Hint Loss: 1.7799
Epoch 3/15 - Hint Loss: 1.6227
Epoch 4/15 - Hint Loss: 1.5381
Epoch 5/15 - Hint Loss: 1.4858
Epoch 6/15 - Hint Loss: 1.4503
Epoch 7/15 - Hint Loss: 1.4265
Epoch 8/15 - Hint Loss: 1.4096
Epoch 9/15 - Hint Loss: 1.3952
Epoch 10/15 - Hint Loss: 1.3828
Epoch 11/15 - Hint Loss: 1.3746
Epoch 12/15 - Hint Loss: 1.3665
Epoch 13/15 - Hint Loss: 1.3610
Epoch 14/15 - Hint Loss: 1.3567
Epoch 15/15 - Hint Loss: 1.3537


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 55.1315%, Train Hard Loss: 1.3960, Train Soft Loss: 0.6791, Train Distill Loss: 3.2901 | Val Hard Loss: 1.9760, Val Soft Loss: 0.5630, Val Distill Loss: 5.4924, Val Acc: 51.1990%
Epoch 2/15 - Train Acc: 75.8678%, Train Hard Loss: 0.8345, Train Soft Loss: 0.3656, Train Distill Loss: 1.8374 | Val Hard Loss: 0.8604, Val Soft Loss: 0.2325, Val Distill Loss: 2.2904, Val Acc: 76.6787%
Epoch 3/15 - Train Acc: 83.5462%, Train Hard Loss: 0.5655, Train Soft Loss: 0.2536, Train Distill Loss: 1.2640 | Val Hard Loss: 0.7132, Val Soft Loss: 0.2123, Val Distill Loss: 2.0551, Val Acc: 79.9161%
Epoch 4/15 - Train Acc: 87.6183%, Train Hard Loss: 0.3915, Train Soft Loss: 0.1990, Train Distill Loss: 0.9499 | Val Hard Loss: 0.6413, Val Soft Loss: 0.1966, Val Distill Loss: 1.8938, Val Acc: 83.8729%
Epoch 5/15 - Train Acc: 92.3065%, Train Hard Loss: 0.2479, Train Soft Loss: 0.1506, Train Distill Loss: 0.6801 | Val Hard Loss: 0.7002, Val Soft Loss: 0.2165, Val Distill Loss: 2.0824, Val

0.9404990403071017

In [13]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.85,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2516
Epoch 2/15 - Hint Loss: 1.7762
Epoch 3/15 - Hint Loss: 1.6183
Epoch 4/15 - Hint Loss: 1.5379
Epoch 5/15 - Hint Loss: 1.4877
Epoch 6/15 - Hint Loss: 1.4525
Epoch 7/15 - Hint Loss: 1.4264
Epoch 8/15 - Hint Loss: 1.4065
Epoch 9/15 - Hint Loss: 1.3916
Epoch 10/15 - Hint Loss: 1.3806
Epoch 11/15 - Hint Loss: 1.3738
Epoch 12/15 - Hint Loss: 1.3656
Epoch 13/15 - Hint Loss: 1.3682
Epoch 14/15 - Hint Loss: 1.3589
Epoch 15/15 - Hint Loss: 1.3538


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 58.1217%, Train Hard Loss: 1.2847, Train Soft Loss: 0.6513, Train Distill Loss: 2.6551 | Val Hard Loss: 1.0518, Val Soft Loss: 0.3465, Val Distill Loss: 3.2977, Val Acc: 69.3046%
Epoch 2/15 - Train Acc: 76.6491%, Train Hard Loss: 0.8000, Train Soft Loss: 0.3690, Train Distill Loss: 1.5656 | Val Hard Loss: 1.0076, Val Soft Loss: 0.3001, Val Distill Loss: 2.9045, Val Acc: 74.7602%
Epoch 3/15 - Train Acc: 83.6664%, Train Hard Loss: 0.5515, Train Soft Loss: 0.2688, Train Distill Loss: 1.1140 | Val Hard Loss: 0.6921, Val Soft Loss: 0.2270, Val Distill Loss: 2.1618, Val Acc: 81.2950%
Epoch 4/15 - Train Acc: 89.0308%, Train Hard Loss: 0.3596, Train Soft Loss: 0.1984, Train Distill Loss: 0.7818 | Val Hard Loss: 0.5312, Val Soft Loss: 0.1722, Val Distill Loss: 1.6431, Val Acc: 85.9113%
Epoch 5/15 - Train Acc: 93.6890%, Train Hard Loss: 0.2083, Train Soft Loss: 0.1470, Train Distill Loss: 0.5297 | Val Hard Loss: 0.3977, Val Soft Loss: 0.1427, Val Distill Loss: 1.3407, Val

0.9380998080614203

In [14]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.90,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2386
Epoch 2/15 - Hint Loss: 1.7779
Epoch 3/15 - Hint Loss: 1.6136
Epoch 4/15 - Hint Loss: 1.5326
Epoch 5/15 - Hint Loss: 1.4835
Epoch 6/15 - Hint Loss: 1.4493
Epoch 7/15 - Hint Loss: 1.4245
Epoch 8/15 - Hint Loss: 1.4059
Epoch 9/15 - Hint Loss: 1.3904
Epoch 10/15 - Hint Loss: 1.3795
Epoch 11/15 - Hint Loss: 1.3714
Epoch 12/15 - Hint Loss: 1.3639
Epoch 13/15 - Hint Loss: 1.3588
Epoch 14/15 - Hint Loss: 1.3538
Epoch 15/15 - Hint Loss: 1.3500


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 55.3569%, Train Hard Loss: 1.3173, Train Soft Loss: 0.7053, Train Distill Loss: 2.3141 | Val Hard Loss: 1.1701, Val Soft Loss: 0.4343, Val Distill Loss: 4.0594, Val Acc: 63.2494%
Epoch 2/15 - Train Acc: 74.9812%, Train Hard Loss: 0.7906, Train Soft Loss: 0.4080, Train Distill Loss: 1.3644 | Val Hard Loss: 1.1197, Val Soft Loss: 0.3742, Val Distill Loss: 3.5533, Val Acc: 69.9041%
Epoch 3/15 - Train Acc: 82.2990%, Train Hard Loss: 0.5648, Train Soft Loss: 0.3001, Train Distill Loss: 0.9886 | Val Hard Loss: 1.0647, Val Soft Loss: 0.3359, Val Distill Loss: 3.2192, Val Acc: 72.7218%
Epoch 4/15 - Train Acc: 87.1375%, Train Hard Loss: 0.3950, Train Soft Loss: 0.2345, Train Distill Loss: 0.7306 | Val Hard Loss: 0.8346, Val Soft Loss: 0.2688, Val Distill Loss: 2.5679, Val Acc: 77.9976%
Epoch 5/15 - Train Acc: 91.9459%, Train Hard Loss: 0.2628, Train Soft Loss: 0.1813, Train Distill Loss: 0.5266 | Val Hard Loss: 0.5262, Val Soft Loss: 0.1772, Val Distill Loss: 1.6808, Val

0.9309021113243762